# Global Settings

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
import random
import os
import warnings
import json
warnings.filterwarnings('ignore')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

In [ ]:
def stratified_global_group_split(df, group_col="name", stratify_col="task_id",
                                  test_size=0.2, random_state=42):
    """
    Globally partition the dataset into train/test groups based on compound name.
    Use a distribution-matching greedy algorithm to select the test set
    to ensure the distribution of each task in the train and val is similar.
    """
    rng = np.random.RandomState(random_state)

    compound_task = (
        df.groupby([group_col, stratify_col])
        .size()
        .unstack(fill_value=0)
        .astype(bool)
        .astype(int)
    )

    n_compounds = len(compound_task)
    n_test = max(1, int(n_compounds * test_size))
    compound_names = compound_task.index.to_numpy()

    total_vec = compound_task.sum().values
    target_prop = total_vec / total_vec.sum()

    remaining = set(compound_names.tolist())
    test_set = set()
    test_vec = np.zeros(len(total_vec))

    for _ in range(n_test):
        n_candidates = max(1, len(remaining) // 5)
        candidates = rng.choice(list(remaining), n_candidates, replace=False)

        best_name, best_score = None, float("inf")
        for name in candidates:
            proposed = test_vec + compound_task.loc[name].values
            s = proposed.sum()
            if s == 0:
                continue
            proposed_prop = proposed / s
            score = ((proposed_prop - target_prop) ** 2).sum()
            if score < best_score:
                best_score = score
                best_name = name

        test_set.add(best_name)
        remaining.remove(best_name)
        test_vec += compound_task.loc[best_name].values

    train_names = set(compound_names.tolist()) - test_set
    train_df = df[df[group_col].isin(train_names)].reset_index(drop=True)
    test_df  = df[df[group_col].isin(test_set)].reset_index(drop=True)

    return train_df, test_df

def group_kfold(df, group_col, stratify_col, n_splits=5, random_state=42):
    rng = np.random.RandomState(random_state)
    compound_names = df[group_col].unique()
    rng.shuffle(compound_names)
    fold_compounds = [set() for _ in range(n_splits)]
    for i, name in enumerate(compound_names):
        fold_compounds[i % n_splits].add(name)
    all_tasks = set(df[stratify_col].unique())
    # to ensure each task appears in both the train and val
    changed = True
    max_iter = 200
    iteration = 0
    while changed and iteration < max_iter:
        changed = False
        iteration += 1
        for fold_idx in range(n_splits):
            val_names = fold_compounds[fold_idx]
            train_names = set(compound_names) - val_names
            val_tasks = set(df[df[group_col].isin(val_names)][stratify_col].unique())
            train_tasks = set(df[df[group_col].isin(train_names)][stratify_col].unique())
            # Fix missing tasks in val
            for task_id in (all_tasks - val_tasks):
                candidates = df[
                    (df[stratify_col] == task_id) &
                    (df[group_col].isin(train_names))
                ][group_col].unique()
                if len(candidates) > 0:
                    compound_to_move = rng.choice(candidates)
                    fold_compounds[fold_idx].add(compound_to_move)
                    train_names.remove(compound_to_move)
                    changed = True
            # Fix missing tasks in train
            for task_id in (all_tasks - train_tasks):
                candidates = df[
                    (df[stratify_col] == task_id) &
                    (df[group_col].isin(val_names))
                ][group_col].unique()
                if len(candidates) > 0:
                    compound_to_move = rng.choice(candidates)
                    fold_compounds[fold_idx].remove(compound_to_move)
                    train_names.add(compound_to_move)
                    changed = True

    kfold_splits = []
    for fold_idx in range(n_splits):
        val_names = fold_compounds[fold_idx]
        train_names = set(compound_names) - val_names
        val_mask = df[group_col].isin(val_names)
        train_mask = df[group_col].isin(train_names)
        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]
        kfold_splits.append((train_idx, val_idx))
    return kfold_splits

# Dataset

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, df):
        self.fp = df.iloc[:, :5000].values.astype("float32")
        self.species = df["species_id"].values.astype("int64")
        self.route = df["route_id"].values.astype("int64")
        self.task = df["task_id"].values.astype("int64")
        self.y = df["ld50"].values.astype("float32")
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.fp[idx]),
            torch.tensor(self.species[idx]),
            torch.tensor(self.route[idx]),
            torch.tensor(self.task[idx]),
            torch.tensor(self.y[idx])
        )

# Endpoint Graph Construction

In [ ]:
def build_endpoint_graph():
    graph_pairs = []
    for i in range(12):
        for j in range(i + 1, 12):
            species_i, route_i = i // 2, i % 2
            species_j, route_j = j // 2, j % 2
            if species_i == species_j or route_i == route_j:
                graph_pairs.append((i, j))
    return graph_pairs

# LOSS Function

In [ ]:
def compute_task_weights(task_ids, num_tasks=12):
    counts = np.bincount(task_ids, minlength=num_tasks)
    base_weights = 1.0 / np.log1p(counts)
    base_weights = base_weights / base_weights.sum() * num_tasks
    return torch.tensor(base_weights, dtype=torch.float32)

def compute_adaptive_task_weights(task_r2_scores, task_counts, num_tasks=12):
    weights = []
    for r2, count in zip(task_r2_scores, task_counts):
        if r2 < 0.3:
            r2_weight = 5.0 
        elif r2 < 0.4:
            r2_weight = 4.0 
        elif r2 < 0.5:
            r2_weight = 3.0  
        elif r2 < 0.6:
            r2_weight = 2.0  
        elif r2 < 0.7:
            r2_weight = 1.0  
        else:
            r2_weight = 0.5  
        if count < 20:
            sample_weight = 3.0 
        elif count < 50:
            sample_weight = 2.0  
        elif count < 100:
            sample_weight = 1.5  
        else:
            sample_weight = 1.0 
        combined_weight = r2_weight * sample_weight
        weights.append(combined_weight)
    weights = np.array(weights)
    weights = weights / weights.sum() * num_tasks
    return torch.tensor(weights, dtype=torch.float32)

def graph_loss(task_embeddings, task_id, graph_pairs):
    loss = 0.0
    count = 0
    for i, j in graph_pairs:
        mask_i = (task_id == i)
        mask_j = (task_id == j)
        if mask_i.sum() > 0 and mask_j.sum() > 0:
            emb_i = task_embeddings[mask_i].mean(0)
            emb_j = task_embeddings[mask_j].mean(0)
            loss += F.mse_loss(emb_i, emb_j)
            count += 1 
    return loss / max(count, 1)


def ordinal_loss(pred_logits, target, bins):
    device = pred_logits.device
    bins = bins.to(device)
    sorted_bins = torch.sort(bins)[0]
    class_idx = torch.zeros(target.size(0), dtype=torch.long, device=device)
    class_idx[target.squeeze() < sorted_bins[0]] = 0 
    class_idx[(target.squeeze() >= sorted_bins[0]) & (target.squeeze() < sorted_bins[1])] = 1 
    class_idx[target.squeeze() >= sorted_bins[1]] = 2 
    return F.cross_entropy(pred_logits, class_idx)


def shrinkage_loss(pred, target, task_id, task_means, shrinkage_factor=0.5):
    device = pred.device
    task_means = task_means.to(device)
    shrinkage_target = (1 - shrinkage_factor) * target + shrinkage_factor * task_means[task_id].unsqueeze(1)
    return F.mse_loss(pred, shrinkage_target)


class TaskWeightedMSE(nn.Module):
    def __init__(self, task_weights, adaptive_weights=None):
        super().__init__()
        self.register_buffer('task_weights', task_weights)
        if adaptive_weights is not None:
            self.register_buffer('adaptive_weights', adaptive_weights)
        else:
            self.adaptive_weights = None
    
    def forward(self, pred, target, task_id):
        mse = (pred - target) ** 2
        if self.adaptive_weights is not None:
            w = self.adaptive_weights[task_id].unsqueeze(1)
        else:
            w = self.task_weights[task_id].unsqueeze(1)
        return (w * mse).mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, pred, target):
        residual = (pred - target) ** 2
        weight = (residual ** self.gamma) + self.alpha
        loss = weight * residual
        return loss.mean()


def compute_combined_loss(pred, pred_logits, target, task_id, task_embeddings, 
                          loss_fn, focal_loss=None, use_focal=False,
                          graph_pairs=None, w_graph=0.1,
                          task_means=None, w_shrinkage=0.3, shrinkage_factor=0.5,
                          w_ordinal=0.2, ordinal_bins=None):
    task_weighted_mse = loss_fn(pred, target, task_id)

    if use_focal and focal_loss is not None:
        focal_loss_value = focal_loss(pred, target)
        total_loss = 0.7 * task_weighted_mse + 0.3 * focal_loss_value
    else:
        total_loss = task_weighted_mse

    if graph_pairs is not None and w_graph > 0:
        graph_loss_value = graph_loss(task_embeddings, task_id, graph_pairs)
        total_loss += w_graph * graph_loss_value

    if task_means is not None and w_shrinkage > 0:
        shrinkage_loss_value = shrinkage_loss(pred, target, task_id, task_means, shrinkage_factor)
        total_loss += w_shrinkage * shrinkage_loss_value
    
    if pred_logits is not None and w_ordinal > 0 and ordinal_bins is not None:
        ordinal_loss_value = ordinal_loss(pred_logits, target, ordinal_bins)
        total_loss += w_ordinal * ordinal_loss_value
    
    return total_loss

In [ ]:
def build_loader(dataset, task_ids, batch_size=64):
    counts = np.bincount(task_ids)
    sample_weights = 1.0 / counts[task_ids]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    return loader

# ChemicalEncoder

In [ ]:
class ChemicalEncoder(nn.Module):
    def __init__(self, in_dim=5000, hidden=512, out_dim=128, dropout1=0.4, dropout2=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout1),
            nn.Linear(hidden, out_dim),
            nn.Dropout(dropout2)
        )
    def forward(self, x):
        return self.net(x)

# ConditionEncoder

In [ ]:
class ConditionEncoder(nn.Module):
    def __init__(self, species_num=6, route_num=2, emb_dim=32, out_dim=128, dropout=0.2):
        super().__init__()
        self.species_emb = nn.Embedding(species_num, emb_dim)
        self.route_emb = nn.Embedding(route_num, 8)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + 8 + emb_dim * 8, 128),  
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, out_dim)
        )
    
    def forward(self, species_id, route_id):
        s = self.species_emb(species_id)  
        r = self.route_emb(route_id)    
        interaction = torch.einsum('bi,bj->bij', s, r).view(s.size(0), -1) 
        cond = torch.cat([s, r, interaction], dim=1)  
        return self.mlp(cond)

# CrossAttentionLayer

In [ ]:
class CrossAttentionLayer(nn.Module):
    def __init__(self, dim=128, num_heads=2, dropout=0.3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)
        
    def forward(self, z_chem, z_cond):
        batch_size = z_chem.size(0)
        residual = z_chem
        z_chem = z_chem.unsqueeze(1)
        z_cond = z_cond.unsqueeze(1)
        Q = self.q_proj(z_cond)
        K = self.k_proj(z_chem)
        V = self.v_proj(z_chem)
        Q = Q.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = torch.matmul(attn_weights, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1)
        out = self.out_proj(out)
        out = self.norm(residual + out)
        return out

# MMoE

In [ ]:
class Expert(nn.Module):
    def __init__(self, dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class MMoE(nn.Module):
    def __init__(self, dim=128, num_experts=2, num_tasks=12):
        super().__init__()
        self.experts = nn.ModuleList([Expert(dim) for _ in range(num_experts)])
        self.gates = nn.ModuleList([nn.Linear(dim, num_experts) for _ in range(num_tasks)])
    def forward(self, x, task_id):
        expert_outs = torch.stack([e(x) for e in self.experts], dim=1)
        outputs = []
        for i in range(len(self.gates)):
            mask = (task_id == i)
            if mask.sum() == 0:
                continue
            gate_w = F.softmax(self.gates[i](x[mask]), dim=1)
            mixed = torch.sum(gate_w.unsqueeze(2) * expert_outs[mask], dim=1)
            outputs.append((mask, mixed))
        if outputs:
            final = torch.zeros_like(x)
            for mask, out in outputs:
                final[mask] = out
            return final
        else:
            return x

# TaskEmbeddingLayer 

In [ ]:
class TaskEmbeddingLayer(nn.Module):
    def __init__(self, num_tasks=12, emb_dim=32):
        super().__init__()
        self.task_emb = nn.Embedding(num_tasks, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, emb_dim)
        )
    
    def forward(self, task_id):
        emb = self.task_emb(task_id)
        return self.mlp(emb)

# AdaptiveTaskHeads

In [ ]:
class AdaptiveTaskHeads(nn.Module):
    def __init__(self, dim=128, num_tasks=12, dropout=0.2, 
                 use_shrinkage=True, use_ordinal=True, device=None):
        super().__init__()
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.device = device
        self.num_tasks = num_tasks
        self.use_shrinkage = use_shrinkage
        self.use_ordinal = use_ordinal
        self.register_buffer('low_performance_tasks', torch.tensor([], dtype=torch.long, device=device))
        self.register_buffer('tiny_sample_tasks', torch.tensor([], dtype=torch.long, device=device))
        self.normal_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
            )
            for _ in range(num_tasks)
        ])
        self.advanced_heads = nn.ModuleDict({})
        self.tiny_sample_heads = nn.ModuleDict({})
        if self.use_shrinkage:
            self.shrinkage_head = nn.Linear(dim, 1)
            self.shrinkage_factor = nn.Parameter(torch.tensor(0.5))
        if self.use_ordinal:
            self.ordinal_head = nn.Linear(dim, 3)
            self.ordinal_bins = nn.Parameter(torch.tensor([2.0, 3.0]))
    
    def update_task_categories(self, task_r2_scores, task_counts):
        r2_tensor = torch.tensor(task_r2_scores)
        count_tensor = torch.tensor(task_counts)
        low_perf_mask = r2_tensor < 0.5
        low_perf_tasks = torch.nonzero(low_perf_mask).flatten().tolist()
        tiny_sample_mask = count_tensor < 20
        tiny_sample_tasks = torch.nonzero(tiny_sample_mask).flatten().tolist()
        self.register_buffer('low_performance_tasks', torch.tensor(low_perf_tasks, dtype=torch.long, device=self.device))
        self.register_buffer('tiny_sample_tasks', torch.tensor(tiny_sample_tasks, dtype=torch.long, device=self.device))
        for task_id in low_perf_tasks:
            if str(task_id) not in self.advanced_heads:
                self.advanced_heads[str(task_id)] = self._create_advanced_head(
                    self.normal_heads[0][0].in_features
                )
                self.advanced_heads[str(task_id)] = self.advanced_heads[str(task_id)].to(self.device)
        for task_id in tiny_sample_tasks:
            if str(task_id) not in self.tiny_sample_heads:
                self.tiny_sample_heads[str(task_id)] = self._create_tiny_sample_head(
                    self.normal_heads[0][0].in_features
                )
                self.tiny_sample_heads[str(task_id)] = self.tiny_sample_heads[str(task_id)].to(self.device)
        print(f"R² < 0.5: {low_perf_tasks}")
        print(f"samples < 20): {tiny_sample_tasks}")
    
    def _create_advanced_head(self, dim):
        return nn.Sequential(
            nn.Linear(dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )
    
    def _create_tiny_sample_head(self, dim):
        return nn.Sequential(
            nn.Linear(dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x, task_id, return_all=False):
        reg_pred = torch.zeros(x.size(0), 1, device=x.device)
        for i in range(len(self.normal_heads)):
            mask = (task_id == i)
            if mask.sum() > 0:
                is_low_perf = i in self.low_performance_tasks.cpu().tolist()
                is_tiny_sample = i in self.tiny_sample_tasks.cpu().tolist()
                if is_tiny_sample and str(i) in self.tiny_sample_heads:
                    task_pred = self.tiny_sample_heads[str(i)](x[mask])
                elif is_low_perf and str(i) in self.advanced_heads:
                    task_pred = self.advanced_heads[str(i)](x[mask])
                else:
                    task_pred = self.normal_heads[i](x[mask])
                if self.use_shrinkage and (is_low_perf or is_tiny_sample):
                    global_pred = self.shrinkage_head(x[mask])
                    if is_tiny_sample:
                        shrink_weight = torch.sigmoid(self.shrinkage_factor) * 0.8 + 0.2
                    else:
                        shrink_weight = torch.sigmoid(self.shrinkage_factor)
                    reg_pred[mask] = (1 - shrink_weight) * task_pred + shrink_weight * global_pred
                else:
                    reg_pred[mask] = task_pred
        if self.use_ordinal and return_all:
            ordinal_logits = self.ordinal_head(x)
            return reg_pred, ordinal_logits, self.ordinal_bins
        else:
            return reg_pred

# ImprovedToxicityModel

In [ ]:
class ImprovedToxicityModel(nn.Module):
    def __init__(self, device='cuda', dropout_chem1=0.4, dropout_chem2=0.3, 
                 dropout_cond=0.2, dropout_attn=0.3, dropout_expert=0.2, dropout_head=0.2,
                 num_heads=2, num_experts=2,
                 use_shrinkage=True, use_ordinal=True):
        super().__init__()
        self.chem_encoder = ChemicalEncoder(dropout1=dropout_chem1, dropout2=dropout_chem2)
        self.cond_encoder = ConditionEncoder(dropout=dropout_cond)
        self.cross_attn = CrossAttentionLayer(dim=128, num_heads=num_heads, dropout=dropout_attn)
        self.mmoe = MMoE(num_experts=num_experts)
        self.task_embedding = TaskEmbeddingLayer() 
        self.heads = AdaptiveTaskHeads(dropout=dropout_head, 
                                       use_shrinkage=use_shrinkage,
                                       use_ordinal=use_ordinal,
                                       device=device)
    
    def forward(self, fp, species_id, route_id, task_id, return_embeddings=False):
        z_chem = self.chem_encoder(fp)
        z_cond = self.cond_encoder(species_id, route_id)
        z = self.cross_attn(z_chem, z_cond)
        z = self.mmoe(z, task_id)
        task_embeddings = self.task_embedding(task_id)
        reg_pred, ordinal_logits, ordinal_bins = self.heads(z, task_id, return_all=True)
        if return_embeddings:
            return reg_pred, ordinal_logits, task_embeddings, ordinal_bins
        else:
            return reg_pred, ordinal_logits, ordinal_bins

# Loading Data

In [ ]:
df = pd.read_csv("all_species_route_summary.csv")
train_df, test_df = stratified_global_group_split(df, group_col='name', stratify_col='task_id',
                                                  test_size=0.2, random_state=42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Compute Task Means

In [ ]:
def compute_task_means(df):
    task_means = []
    for task_id in range(12):
        task_df = df[df['task_id'] == task_id]
        mean_value = task_df['ld50'].mean()
        task_means.append(mean_value)
    return torch.tensor(task_means, dtype=torch.float32)

task_means = compute_task_means(train_df)
task_names = ['mouse_oral', 'mouse_iv', 'rat_oral', 'rat_iv', 'rabbit_oral', 'rabbit_iv',
              'dog_oral', 'dog_iv', 'cat_oral', 'cat_iv', 'guineapig_oral', 'guineapig_iv']
for i, (name, mean) in enumerate(zip(task_names, task_means)):
    print(f" {name} (Task {i}): {mean:.4f}")

# Bayesian Optimization 

In [ ]:
def objective(trial, train_df, n_folds=3, max_epochs=50, device='cuda', use_focal=True):
    adaptive_weights = torch.ones(12, dtype=torch.float32).to(device)
    task_means = compute_task_means(train_df).to(device)
    graph_pairs = build_endpoint_graph()
    params = {
        'lr': trial.suggest_float('lr', 1e-5, 1e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'dropout_chem1': trial.suggest_float('dropout_chem1', 0.2, 0.6),
        'dropout_chem2': trial.suggest_float('dropout_chem2', 0.1, 0.5),
        'dropout_cond': trial.suggest_float('dropout_cond', 0.1, 0.4),
        'dropout_attn': trial.suggest_float('dropout_attn', 0.1, 0.5),
        'dropout_expert': trial.suggest_float('dropout_expert', 0.1, 0.4),
        'dropout_head': trial.suggest_float('dropout_head', 0.1, 0.4),
        'num_heads': trial.suggest_categorical('num_heads', [1, 2, 4]),
        'num_experts': trial.suggest_categorical('num_experts', [2, 3, 4, 6]),
        'patience': trial.suggest_int('patience', 10, 30),
        'T_0': trial.suggest_int('T_0', 10, 50),
        'w_graph': trial.suggest_float('w_graph', 0.0, 0.5),
        'w_shrinkage': trial.suggest_float('w_shrinkage', 0.0, 0.5),
        'shrinkage_factor': trial.suggest_float('shrinkage_factor', 0.0, 1.0),
        'w_ordinal': trial.suggest_float('w_ordinal', 0.0, 0.5),
    }
    kfold_splits = group_kfold(train_df, group_col='name', stratify_col='task_id',
                                         n_splits=n_folds, random_state=42)
    fold_r2_scores = []
    for fold, (train_idx, val_idx) in enumerate(kfold_splits):
        set_seed(42 + fold)
        train_df_fold = train_df.iloc[train_idx].reset_index(drop=True)
        val_df_fold = train_df.iloc[val_idx].reset_index(drop=True)
        train_dataset = ToxicityDataset(train_df_fold)
        val_dataset = ToxicityDataset(val_df_fold)
        batch_size = params['batch_size']
        train_loader = build_loader(train_dataset, train_df_fold['task_id'].values, batch_size=batch_size)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        base_weights = compute_task_weights(train_df_fold['task_id'].values).to(device)
        task_weighted_mse = TaskWeightedMSE(base_weights, adaptive_weights).to(device)
        focal_loss = FocalLoss(gamma=2.0, alpha=1.0) if use_focal else None
        model = ImprovedToxicityModel(
            dropout_chem1=params['dropout_chem1'],
            dropout_chem2=params['dropout_chem2'],
            dropout_cond=params['dropout_cond'],
            dropout_attn=params['dropout_attn'],
            dropout_expert=params['dropout_expert'],
            dropout_head=params['dropout_head'],
            num_heads=params['num_heads'],
            num_experts=params['num_experts'],
            use_shrinkage=True,
            use_ordinal=True
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), 
                                       lr=params['lr'], 
                                       weight_decay=params['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=params['T_0'], T_mult=2, eta_min=1e-6
        )
        patience = params['patience']
        best_val_loss = float('inf')
        patience_counter = 0
        for epoch in range(max_epochs):
            model.train()
            for fp, species, route, task, y in train_loader:
                fp = fp.to(device)
                species = species.to(device)
                route = route.to(device)
                task = task.to(device)
                y = y.unsqueeze(1).to(device)
                pred, pred_logits, task_embeddings, ordinal_bins = model(fp, species, route, task, return_embeddings=True)
                loss = compute_combined_loss(
                    pred, pred_logits, y, task, task_embeddings,
                    task_weighted_mse, focal_loss, use_focal,
                    graph_pairs=graph_pairs, 
                    w_graph=params['w_graph'],
                    task_means=task_means, 
                    w_shrinkage=params['w_shrinkage'], 
                    shrinkage_factor=params['shrinkage_factor'],
                    w_ordinal=params['w_ordinal'],
                    ordinal_bins=ordinal_bins
                )
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            val_tasks = []  
            with torch.no_grad():
                for fp, species, route, task, y in val_loader:
                    fp = fp.to(device)
                    species = species.to(device)
                    route = route.to(device)
                    task = task.to(device)
                    y = y.unsqueeze(1).to(device)
                    pred, pred_logits, task_embeddings, ordinal_bins = model(fp, species, route, task, return_embeddings=True)
                    loss = compute_combined_loss(
                        pred, pred_logits, y, task, task_embeddings,
                        task_weighted_mse, focal_loss, use_focal,
                        graph_pairs=graph_pairs, 
                        w_graph=params['w_graph'],
                        task_means=task_means, 
                        w_shrinkage=params['w_shrinkage'], 
                        shrinkage_factor=params['shrinkage_factor'],
                        w_ordinal=params['w_ordinal'],
                        ordinal_bins=ordinal_bins
                    )
                    val_loss += loss.item()
                    val_preds.extend(pred.cpu().numpy().flatten())
                    val_labels.extend(y.cpu().numpy().flatten())
                    val_tasks.extend(task.cpu().numpy()) 
            avg_val_loss = val_loss / len(val_loader)
            scheduler.step()
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= patience:
                break
        val_r2 = r2_score(val_labels, val_preds)
        fold_r2_scores.append(val_r2)
        task_r2_scores = []
        task_counts = []
        for task_id in range(12):
            task_mask = np.array(val_tasks) == task_id
            if task_mask.sum() > 0:
                task_preds = np.array(val_preds)[task_mask]
                task_labels = np.array(val_labels)[task_mask]
                task_r2 = r2_score(task_labels, task_preds)
                task_r2_scores.append(task_r2)
                task_counts.append(task_mask.sum())
            else:
                task_r2_scores.append(0.0) 
                task_counts.append(0) 
        adaptive_weights = compute_adaptive_task_weights(np.array(task_r2_scores),np.array(task_counts)).to(device)
        model.heads.update_task_categories(task_r2_scores, task_counts)
    mean_r2 = np.mean(fold_r2_scores)
    return mean_r2

In [ ]:
def run_bayesian_optimization(train_df, n_trials=30, n_folds=3, max_epochs=50, device='cuda', use_focal=True):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    def objective_wrapper(trial):
        return objective(trial, train_df, n_folds=n_folds, max_epochs=max_epochs, device=device, use_focal=use_focal)
    study.optimize(
        objective_wrapper,
        n_trials=n_trials,
        show_progress_bar=True,
        callbacks=[
            lambda study, trial: print(
                f"Trial {trial.number}: R² = {trial.value:.4f}, "
                f"w_graph={trial.params['w_graph']:.3f}, "
                f"w_shrinkage={trial.params['w_shrinkage']:.3f}, "
                f"w_ordinal={trial.params['w_ordinal']:.3f}"
            )
        ]
    )
    for key, value in study.best_params.items():
        print(f" {key}: {value}")
    print(f"{'='*80}\n")
    return study.best_params, study.best_value, study

In [ ]:
best_params = {
    'lr': 5e-4,
    'weight_decay': 1e-2,
    'batch_size': 64,
    'dropout_chem1': 0.4,
    'dropout_chem2': 0.3,
    'dropout_cond': 0.2,
    'dropout_attn': 0.3,
    'dropout_expert': 0.2,
    'dropout_head': 0.2,
    'num_heads': 2,
    'num_experts': 2,
    'patience': 15,
    'T_0': 30,
    'w_graph': 0.1,
    'w_shrinkage': 0.3,
    'shrinkage_factor': 0.5,
    'w_ordinal': 0.2,
}
RUN_OPTUNA_OPTIMIZATION = True
if RUN_OPTUNA_OPTIMIZATION:
    optimized_params, best_r2, study = run_bayesian_optimization(
        train_df,
        n_trials=30,
        n_folds=3,
        max_epochs=50,
        device=device,
        use_focal=True
    )
    best_params = optimized_params
    optuna_results = {
        'best_params': best_params,
        'best_r2': best_r2,
        'n_trials': len(study.trials)
    }
    with open('optuna_results.json', 'w') as f:
        json.dump(optuna_results, f, indent=2)
for key, value in best_params.items():
    print(f"  {key}: {value}")

# K-fold Cross Validation

In [ ]:
def train_with_cross_validation(df, n_splits=5, device='cuda', best_params=None, use_focal=True):
    kfold_splits = group_kfold(df, group_col='name', stratify_col='task_id',
                                         n_splits=n_splits, random_state=42)
    fold_results = []
    all_models = []
    all_train_losses = []
    all_val_losses = []
    all_fold_r2_scores = []
    all_fold_task_counts = []
    task_names = ['mouse_oral', 'mouse_intravenous', 'rat_oral', 'rat_intravenous', 
                  'rabbit_oral', 'rabbit_intravenous', 'dog_oral', 'dog_intravenous', 
                  'cat_oral', 'cat_intravenous', 'guineapig_oral', 'guineapig_intravenous']
    adaptive_weights = torch.ones(12, dtype=torch.float32).to(device)
    for i, (name, w) in enumerate(zip(task_names, adaptive_weights)):
        print(f" {name} (Task {i}): {w:.4f}")
    base_weights = compute_task_weights(df['task_id'].values).to(device)
    global_task_weighted_mse = TaskWeightedMSE(base_weights, adaptive_weights).to(device)   
    for fold, (train_idx, val_idx) in enumerate(kfold_splits):
        print(f"\n{'='*60}")
        print(f"Train Fold {fold + 1}/{n_splits}")
        print(f"{'='*60}")
        train_df_fold = df.iloc[train_idx].reset_index(drop=True)
        val_df_fold = df.iloc[val_idx].reset_index(drop=True)
        if len(all_fold_r2_scores) > 0:
            avg_r2 = np.mean(all_fold_r2_scores, axis=0)
            avg_counts = np.mean(all_fold_task_counts, axis=0)
            adaptive_weights = compute_adaptive_task_weights(avg_r2, avg_counts).to(device)
            print(f"\n use {len(all_fold_r2_scores)} historical folds to compute adaptive weights")
        else:
            adaptive_weights = torch.ones(12, dtype=torch.float32).to(device)
            print(f"\n use equal weights (first fold)")
        print(f"\nFold {fold + 1} task weights：")
        for i, (name, w) in enumerate(zip(task_names, adaptive_weights)):
            print(f"  {name} (Task {i}): {w:.4f}")
        global_task_weighted_mse.adaptive_weights = adaptive_weights
        train_dataset = ToxicityDataset(train_df_fold)
        val_dataset = ToxicityDataset(val_df_fold)
        batch_size = best_params['batch_size']
        train_loader = build_loader(train_dataset, train_df_fold['task_id'].values, batch_size=batch_size)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        focal_loss = FocalLoss(gamma=2.0, alpha=1.0) if use_focal else None
        task_means = compute_task_means(train_df_fold).to(device)
        graph_pairs = build_endpoint_graph()
        model = ImprovedToxicityModel(
            device=device,
            dropout_chem1=best_params['dropout_chem1'],
            dropout_chem2=best_params['dropout_chem2'],
            dropout_cond=best_params['dropout_cond'],
            dropout_attn=best_params['dropout_attn'],
            dropout_expert=best_params['dropout_expert'],
            dropout_head=best_params['dropout_head'],
            num_heads=best_params['num_heads'],
            num_experts=best_params['num_experts'],
            use_shrinkage=True,
            use_ordinal=True
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), 
                                       lr=best_params['lr'], 
                                       weight_decay=best_params['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=best_params['T_0'], T_mult=2, eta_min=1e-6
        )
        patience = best_params['patience']
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        train_losses = []
        val_losses = []
        for epoch in range(300):
            model.train()
            total_loss = 0
            for fp, species, route, task, y in train_loader:
                fp, species, route, task, y = fp.to(device), species.to(device), route.to(device), task.to(device), y.unsqueeze(1).to(device)
                pred, pred_logits, task_embeddings, ordinal_bins = model(fp, species, route, task, return_embeddings=True)
                loss = compute_combined_loss(
                    pred, pred_logits, y, task, task_embeddings,
                    global_task_weighted_mse, focal_loss, use_focal=use_focal,
                    graph_pairs=graph_pairs, 
                    w_graph=best_params['w_graph'],
                    task_means=task_means, 
                    w_shrinkage=best_params['w_shrinkage'], 
                    shrinkage_factor=best_params['shrinkage_factor'],
                    w_ordinal=best_params['w_ordinal'],
                    ordinal_bins=ordinal_bins
                )
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
                total_loss += loss.item()
            avg_train_loss = total_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            val_tasks = []
            with torch.no_grad():
                for fp, species, route, task, y in val_loader:
                    fp, species, route, task, y = fp.to(device), species.to(device), route.to(device), task.to(device), y.unsqueeze(1).to(device)
                    pred, pred_logits, task_embeddings, ordinal_bins = model(fp, species, route, task, return_embeddings=True)
                    loss = compute_combined_loss(
                        pred, pred_logits, y, task, task_embeddings,
                        global_task_weighted_mse, focal_loss, use_focal=use_focal,
                        graph_pairs=graph_pairs, 
                        w_graph=best_params['w_graph'],
                        task_means=task_means, 
                        w_shrinkage=best_params['w_shrinkage'], 
                        shrinkage_factor=best_params['shrinkage_factor'],
                        w_ordinal=best_params['w_ordinal'],
                        ordinal_bins=ordinal_bins
                    )
                    val_loss += loss.item()
                    val_preds.extend(pred.cpu().numpy().flatten())
                    val_labels.extend(y.cpu().numpy().flatten())
                    val_tasks.extend(task.cpu().numpy())
            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)
            scheduler.step()
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f" early stopping at epoch {epoch + 1}")
                break
        model.load_state_dict(best_model_state)
        all_models.append(model)
        all_train_losses.append(train_losses)
        all_val_losses.append(val_losses)
        val_rmse = np.sqrt(mean_squared_error(val_labels, val_preds))
        val_mae = mean_absolute_error(val_labels, val_preds)
        val_r2 = r2_score(val_labels, val_preds)
        fold_results.append({
            'fold': fold + 1,
            'val_loss': best_val_loss,
            'val_rmse': val_rmse,
            'val_mae': val_mae,
            'val_r2': val_r2
        })
        print(f"\nFold {fold + 1} results:")
        print(f"  Loss:  {best_val_loss:.4f}")
        print(f"  RMSE:  {val_rmse:.4f}")
        print(f"  MAE:   {val_mae:.4f}")
        print(f"  R²:    {val_r2:.4f}")
    
        task_r2_scores = []
        for task_id in range(12):
            task_mask = np.array(val_tasks) == task_id
            if task_mask.sum() > 0:
                task_preds = np.array(val_preds)[task_mask]
                task_labels = np.array(val_labels)[task_mask]
                task_r2 = r2_score(task_labels, task_preds)
                task_r2_scores.append(task_r2)
            else:
                task_r2_scores.append(0.0)
        print(f"\nFold {fold + 1}task R² scores：")
        for i, r2 in enumerate(task_r2_scores):
            print(f"  {task_names[i]} (Task {i}): R² = {r2:.4f}")
        current_counts = np.bincount(train_df_fold['task_id'].values, minlength=12)
        all_fold_r2_scores.append(task_r2_scores)
        all_fold_task_counts.append(current_counts)

    return fold_results, all_models, all_train_losses, all_val_losses, adaptive_weights

# RUN

In [ ]:
fold_results, all_models, all_train_losses, all_val_losses, adaptive_weights = train_with_cross_validation(
    train_df, 
    n_splits=5,
    device=device, 
    best_params=best_params,
    use_focal=True 
)

In [ ]:
print(f"\n{'='*60}")
print("K fold cross-validation results:")
print(f"{'='*60}")
for result in fold_results:
    print(f"Fold {result['fold']}: R² = {result['val_r2']:.4f}, RMSE = {result['val_rmse']:.4f}")

avg_r2 = np.mean([r['val_r2'] for r in fold_results])
avg_rmse = np.mean([r['val_rmse'] for r in fold_results])
std_r2 = np.std([r['val_r2'] for r in fold_results])

print(f"\nmean performance:")
print(f"mean R²: {avg_r2:.4f} ± {std_r2:.4f}")
print(f"mean RMSE: {avg_rmse:.4f}")

# Test set Evaluation

In [ ]:
def evaluate_ensemble(models, test_df, device):
    test_dataset = ToxicityDataset(test_df)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    all_preds = []
    for model in models:
        model.eval()
        preds = []
        with torch.no_grad():
            for fp, species, route, task, y in test_loader:
                fp, species, route, task = fp.to(device), species.to(device), route.to(device), task.to(device)
                pred, _, _= model(fp, species, route, task)
                preds.append(pred.cpu().numpy())
        all_preds.append(np.concatenate(preds).flatten())
    ensemble_pred = np.mean(all_preds, axis=0)
    return ensemble_pred

In [ ]:
test_predictions = evaluate_ensemble(all_models, test_df, device=device)
test_labels = test_df['ld50'].values
test_rmse = np.sqrt(mean_squared_error(test_labels, test_predictions))
test_mae = mean_absolute_error(test_labels, test_predictions)
test_r2 = r2_score(test_labels, test_predictions)
    
print(f"\n{'='*80}")
print("Test Set Ensemble Evaluation Results")
print(f"{'='*80}")
print(f"Overall Performance:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE: {test_mae:.4f}")
print(f"  R²: {test_r2:.4f}")

# Save Model

In [ ]:
for i, model in enumerate(all_models):
    model_path = f'MSTox_cross_attention_optimized_fold{i+1}.pth'
    test_r2_value = test_r2 if 'test_r2' in locals() else None
    torch.save({
        'model_state_dict': model.state_dict(),
        'best_params': best_params,
        'fold_results': fold_results[i],
        'test_r2': test_r2_value,
        'train_losses': all_train_losses[i],
        'val_losses': all_val_losses[i],
        'fold': i + 1,
    }, model_path)
test_rmse_value = test_rmse if 'test_rmse' in locals() else None
test_mae_value = test_mae if 'test_mae' in locals() else None
test_r2_value = test_r2 if 'test_r2' in locals() else None

ensemble_info = {
    'best_params': best_params,
    'fold_results': fold_results,
    'test_results': {
        'rmse': test_rmse_value,
        'mae': test_mae_value,
        'r2': test_r2_value
    },
    'num_folds': len(all_models)
}
with open('MSTox_cross_attention_optimized_ensemble_info.json', 'w', encoding='utf-8', errors='ignore') as f:
    json.dump(ensemble_info, f, indent=2)